# BCG X - PowerCo Customer Churn Analysis
## Step 3: Exploratory Data Analysis & Data Cleaning
**Analyst:** [Your Name]  
**Date:** 2025  
**Objective:** Understand the datasets provided by PowerCo, explore data types, distributions, and visualize key patterns to inform churn analysis.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded successfully!')

## 2. Load the Datasets
PowerCo provided 3 datasets:
- **client_data.csv** - Historical customer data (usage, sign-up date, forecasted usage, etc.)
- **price_data.csv** - Variable and fixed pricing data at different points in time
- **churn_data.csv** - Churn indicator (whether each customer has churned or not)

In [ ]:
# Load datasets
# NOTE: Update file paths to match where you saved the datasets
client_df   = pd.read_csv('client_data.csv')
price_df    = pd.read_csv('price_data.csv')
churn_df    = pd.read_csv('churn_data.csv')

print(f'Client Data shape   : {client_df.shape}')
print(f'Price Data shape    : {price_df.shape}')
print(f'Churn Data shape    : {churn_df.shape}')

## 3. Data Overview - First Look

In [ ]:
print('=== CLIENT DATA - First 5 rows ===')
client_df.head()

In [ ]:
print('=== PRICE DATA - First 5 rows ===')
price_df.head()

In [ ]:
print('=== CHURN DATA - First 5 rows ===')
churn_df.head()

## 4. Data Types of Each Column

In [ ]:
print('=== CLIENT DATA - Data Types ===')
print(client_df.dtypes)
print()
print('=== PRICE DATA - Data Types ===')
print(price_df.dtypes)
print()
print('=== CHURN DATA - Data Types ===')
print(churn_df.dtypes)

## 5. Descriptive Statistics

In [ ]:
print('=== CLIENT DATA - Descriptive Statistics ===')
client_df.describe(include='all').T

In [ ]:
print('=== PRICE DATA - Descriptive Statistics ===')
price_df.describe(include='all').T

## 6. Missing Values Check

In [ ]:
def check_missing(df, name):
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    result  = pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
    result  = result[result['Missing Count'] > 0].sort_values('Missing %', ascending=False)
    print(f'\n=== {name} - Missing Values ===')
    if result.empty:
        print('No missing values found!')
    else:
        print(result)

check_missing(client_df, 'CLIENT DATA')
check_missing(price_df,  'PRICE DATA')
check_missing(churn_df,  'CHURN DATA')

## 7. Churn Distribution Analysis

In [ ]:
# Identify the churn column (commonly 'churn' or 'churned')
churn_col = 'churn'  # Update this if the column name is different

if churn_col in churn_df.columns:
    churn_counts = churn_df[churn_col].value_counts()
    churn_pct    = churn_df[churn_col].value_counts(normalize=True) * 100

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Bar chart
    axes[0].bar(['Not Churned (0)', 'Churned (1)'], churn_counts, color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[0].set_title('Customer Churn Count', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Customers')
    for i, v in enumerate(churn_counts):
        axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

    # Pie chart
    axes[1].pie(churn_counts, labels=['Not Churned', 'Churned'],
                autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
                startangle=90, explode=(0, 0.05))
    axes[1].set_title('Churn Rate (%)', fontsize=14, fontweight='bold')

    plt.suptitle('PowerCo Customer Churn Distribution', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nChurn Rate: {churn_pct.get(1, churn_pct.iloc[1]):.2f}%')

## 8. Distribution of Numeric Columns - Client Data

In [ ]:
numeric_cols = client_df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns in client data: {numeric_cols}')

# Plot distributions
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(client_df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Numeric Features - Client Data', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Price Analysis - Variable vs Fixed Pricing

In [ ]:
price_numeric = price_df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns in price data: {price_numeric}')

if len(price_numeric) > 0:
    fig, axes = plt.subplots(1, len(price_numeric[:4]), figsize=(15, 4))
    if len(price_numeric[:4]) == 1:
        axes = [axes]

    for i, col in enumerate(price_numeric[:4]):
        axes[i].hist(price_df[col].dropna(), bins=30, color='coral', edgecolor='white', alpha=0.8)
        axes[i].set_title(f'{col}', fontweight='bold')
        axes[i].set_xlabel('Price')
        axes[i].set_ylabel('Frequency')

    plt.suptitle('Price Data Distributions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('price_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Merge Datasets & Churn Analysis by Feature

In [ ]:
# Merge client data with churn indicator
# Assumption: datasets share a common 'id' or 'customer_id' key
# Update key column name as needed

try:
    merged_df = pd.merge(client_df, churn_df, on='id', how='inner')
    print(f'Merged dataset shape: {merged_df.shape}')
    print(merged_df.head())
except KeyError:
    print('Column name mismatch - check the join key in your datasets')
    print('Client columns:', client_df.columns.tolist())
    print('Churn columns :', churn_df.columns.tolist())

## 11. Correlation Heatmap

In [ ]:
try:
    corr_matrix = merged_df.select_dtypes(include=[np.number]).corr()

    plt.figure(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='RdYlGn', center=0, linewidths=0.5,
                annot_kws={'size': 8})
    plt.title('Correlation Heatmap - Merged Dataset', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
except:
    print('Run the merge cell above first before generating the heatmap.')

## 12. Key Observations Summary

In [ ]:
print('=== EDA SUMMARY ===')
print()
print('Dataset Sizes:')
print(f'  - Client data : {client_df.shape[0]} rows, {client_df.shape[1]} columns')
print(f'  - Price data  : {price_df.shape[0]} rows, {price_df.shape[1]} columns')
print(f'  - Churn data  : {churn_df.shape[0]} rows, {churn_df.shape[1]} columns')
print()
print('Next Steps:')
print('  1. Handle missing values (impute or drop)')
print('  2. Convert date columns to datetime format')
print('  3. Engineer new features (e.g., tenure, price sensitivity ratio)')
print('  4. Proceed to Feature Engineering (Step 4)')

---
**End of EDA Notebook**  
Next: Step 4 - Feature Engineering